In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('insurance.csv')
df.head()

In [ ]:
df.columns

## **1. Remove duplicate rows — Data Cleaning Question: Count duplicate rows and remove them**


In [ ]:
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates()

print("Rows remaining:", len(df))

# 2. Check and fill missing numerical values — Data Cleaning Question: Count missing values in every column, then replace missing numeric values with each column's median.**bold text**

In [ ]:
# Question 2: Handle Missing Values

print(df.isnull().sum())

num_cols = df.select_dtypes(include="number").columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print(df[num_cols].isnull().sum())

# 3. Remove invalid values — Data Cleaning Question: Remove rows where BMI or charges is zero or **negative**

In [ ]:
# Question 3: Remove Invalid Values

print(df[["bmi", "charges"]].describe())

df = df[
    (df["bmi"] > 0) &
    (df["charges"] > 0)
].copy()

print("Rows after validity check:", len(df))

# 4. Create a BMI category — Feature Engineering Question: Create BMI_CATEGORY: Underweight below 18.5, Normal from 18.5 to 24.9, Overweight from 25 to 29.9, and Obese at 30 or above.**bold text**

In [ ]:
# Question 4: Create BMI Category

def bmi_category(x):
    if x < 18.5:
        return "Underweight"
    elif x < 25:
        return "Normal"
    elif x < 30:
        return "Overweight"
    return "Obese"

df["BMI_CATEGORY"] = df["bmi"].apply(bmi_category)

print(df[["bmi", "BMI_CATEGORY"]].head())

print(df["BMI_CATEGORY"].value_counts())

# 5. Create a high-charge class — Feature Engineering Question: Create HIGH_CHARGE = 1 when charges are above the median, otherwise 0.**bold text**

In [ ]:
# Question 5: Create High Charge Class

median_charge = df["charges"].median()

df["HIGH_CHARGE"] = (
    df["charges"] > median_charge
).astype(int)

print("Median charge:", median_charge)

print(df["HIGH_CHARGE"].value_counts())

# 6. One-hot encode categorical columns — Encoding Question: Encode sex, smoker, and region using one-hot encoding.**bold text**

In [ ]:
# Question 6: One-Hot Encoding

encoded = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True
)

print(encoded.head())

print([
    c for c in ["sex", "smoker", "region"]
    if c in encoded.columns
])

# 7. Scale numerical features — **Scaling**

In [ ]:
# Question 7: Feature Scaling

from sklearn.preprocessing import StandardScaler

cols = ["age", "bmi", "children"]

X_scaled = StandardScaler().fit_transform(df[cols])

print(X_scaled[:5])

print("Means:", X_scaled.mean(axis=0).round(2))

print("Std devs:", X_scaled.std(axis=0).round(2))

# 8. Build a Linear Regression model — Regression Question: Predict medical charges using age, bmi, and smoker status. Show ve predictions.**bold text**

In [ ]:
# Question 8: Linear Regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df["smoker_yes"] = (df["smoker"] == "yes").astype(int)

X = df[["age", "bmi", "smoker_yes"]]

y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression().fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Actual:", y_test.head().round(2).tolist())

print("Predicted:", lr_pred[:5].round(2).tolist())

# 9. Build a Logistic Regression model — Classi cation Question: Predict whether a person's charges are above the median and print **accuracy**

In [ ]:
# Question 9: Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df["smoker_yes"] = (df["smoker"] == "yes").astype(int)

threshold = df["charges"].median()

X_c = df[["age", "bmi", "smoker_yes"]]

y_c = (df["charges"] > threshold).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42,
    stratify=y_c
)

log_model = LogisticRegression(max_iter=1000).fit(
    X_train_c, y_train_c
)

log_pred = log_model.predict(X_test_c)

print("Accuracy:", round(accuracy_score(y_test_c, log_pred), 3))

# 10. Evaluate Linear Regression with MAE — Regression Metrics Question: Calculate Mean Absolute Error using the predictions[link text]**bold text**

In [ ]:
# Question 10: Calculate MAE

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, lr_pred)

print("MAE:", round(mae, 2))

# 11. Evaluate Linear Regression with RMSE — Regression Metrics Question: Calculate Root Mean Squared Error for the model **ķ**

In [ ]:
# Question 11: Calculate RMSE

from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

print("RMSE:", round(rmse, 2))

## 12. Evaluate Linear Regression with R-squared — Regression Metrics Question: Calculate R-squared for the **prediction**

In [ ]:
# Question 12: Calculate R-Squared

from sklearn.metrics import r2_score

print("R-squared:", round(r2_score(y_test, lr_pred), 3))

# 13. Display a confusion matrix — Classi cation Metrics Question: Create a confusion matrix for the Logistic Regression **mode**

In [ ]:
# Question 13: Confusion Matrix

from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test_c, log_pred))

# 14. Calculate precision, recall, and F1 — Classi cation Metrics Question: Calculate the three metrics for the high-charge class (label 1).**bold text**

In [ ]:
# Question 14: Classification Metrics

from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", round(
    precision_score(y_test_c, log_pred, zero_division=0), 3
))

print("Recall:", round(
    recall_score(y_test_c, log_pred, zero_division=0), 3
))

print("F1:", round(
    f1_score(y_test_c, log_pred, zero_division=0), 3
))

# 16. Plot the distribution of charges — EDA - Plots Question: Draw a histogram and a boxplot of charges. Is the distribution symmetric or skewed?**bold text**

In [ ]:
# Question 16: Distribution of Charges

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].hist(df["charges"], bins=30, edgecolor="black")
ax[0].set_title("Histogram of charges")
ax[0].set_xlabel("Charges (USD)")
ax[0].set_ylabel("Count")

ax[1].boxplot(df["charges"], vert=False)
ax[1].set_title("Boxplot of charges")
ax[1].set_xlabel("Charges (USD)")

plt.tight_layout()
plt.show()

print("Skewness:", round(df["charges"].skew(), 2))

# 17. Sca er plot of BMI vs charges by smoker — EDA - **Plots**

In [ ]:
# Question 17: BMI vs Charges by Smoker

import matplotlib.pyplot as plt

for label, colour in [("no", "tab:blue"), ("yes", "tab:red")]:
    part = df[df["smoker"] == label]

    plt.scatter(
        part["bmi"],
        part["charges"],
        s=12,
        alpha=0.6,
        color=colour,
        label=f"smoker = {label}"
    )

plt.xlabel("BMI")
plt.ylabel("Charges (USD)")
plt.title("BMI vs charges")
plt.legend()
plt.show()

print(df.groupby("smoker")["charges"].mean().round(0))